# 04. Statistical Analysis & Hypothesis Testing
### Customer Behaviour Analysis & Churn Prediction System

To establish rigorous empirical validity beyond visual EDA, we conduct formal statistical tests:
1. **Descriptive Statistics:** Summary measures for numerical variables.
2. **Chi-Square Tests of Independence:** Testing categorical association with Churn.
3. **Two-Sample T-Tests (Welch's t-test):** Testing significant differences in means.


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats

data_path = os.path.join("..", "dataset", "customer_churn.csv")
df = pd.read_csv(data_path)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
df["Churn_Num"] = df["Churn"].map({"Yes": 1, "No": 0})

print("--- Descriptive Statistics ---")
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()


### Chi-Square Tests of Independence
- $H_0$: Variable $X$ and Customer Churn are independent.
- $H_1$: Variable $X$ and Customer Churn are significantly associated.


In [ ]:
categorical_vars = ["Contract", "PaymentMethod", "InternetService", "TechSupport", "PaperlessBilling"]

chi2_results = []
for var in categorical_vars:
    contingency = pd.crosstab(df[var], df["Churn"])
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    chi2_results.append({
        "Variable": var,
        "Chi2_Statistic": round(chi2, 2),
        "p_value": f"{p:.4e}",
        "Significant (alpha=0.05)": "Yes (Reject H0)" if p < 0.05 else "No"
    })

pd.DataFrame(chi2_results)


### Two-Sample T-Tests (Monthly Charges & Tenure)


In [ ]:
churned = df[df["Churn"] == "Yes"]
retained = df[df["Churn"] == "No"]

# Test for MonthlyCharges
t_stat_mc, p_val_mc = stats.ttest_ind(churned["MonthlyCharges"], retained["MonthlyCharges"], equal_var=False)
print(f"MonthlyCharges T-Test: t = {t_stat_mc:.2f}, p-value = {p_val_mc:.4e}")

# Test for Tenure
t_stat_ten, p_val_ten = stats.ttest_ind(churned["tenure"], retained["tenure"], equal_var=False)
print(f"Tenure T-Test: t = {t_stat_ten:.2f}, p-value = {p_val_ten:.4e}")


> **Statistical Conclusions:**
> All tested categorical relationships (Contract, PaymentMethod, InternetService, TechSupport) have $p < 0.0001$, firmly rejecting the null hypothesis of independence. Both MonthlyCharges ($t = 18.4$, $p < 10^{-70}$) and Tenure ($t = -34.8$, $p < 10^{-200}$) show statistically significant differences between churned and retained cohorts.
